# Week 12 Practical — Collaborative & Content-Based Recommender Systems

**Learning objectives:** build a user–item matrix, evaluate a baseline, implement regularized matrix factorization, explore mean normalization and cold start, build content-based representations from movie genres, train a two-tower neural recommender (optional TensorFlow), retrieve similar movies, and evaluate top-K recommendations.

**Data provenance:** Prefer **GroupLens MovieLens Latest Small** (real ratings, genres and tags). The notebook tries local files first and optionally downloads the official archive. If unavailable, it uses the **bundled clearly labeled synthetic movie ratings and profiles**. Synthetic user attributes are *not* part of MovieLens. No actual personal demographics are inferred from MovieLens IDs.

**Student workflow:** Run setup and instructor-provided cells in order. Complete every `TODO` cell, then run its check. Do not fit preprocessing, profiles, or models on validation/test ratings. For a rigorous production benchmark use chronological splits and exposure-aware ranking evaluation; this exercise uses a reproducible per-user random holdout for simplicity.

## Setup and dataset loading
Required: `numpy`, `pandas`, `matplotlib`, `scikit-learn`, and `tensorflow` for the two-tower extension. Run the notebook from its supplied folder or adjust `DATA_DIR`. The downloaded official archive is about 1 MB.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path('week12_data')
if not DATA_DIR.exists():
    DATA_DIR = Path('.')  # when notebook is run from inside the data folder
print('Data folder:', DATA_DIR.resolve())

In [ ]:
# Use official MovieLens files if present; otherwise use the included synthetic fallback.
from urllib.request import urlretrieve
from zipfile import ZipFile
official = DATA_DIR / 'ml-latest-small'
archive = DATA_DIR / 'ml-latest-small.zip'
if not (official / 'ratings.csv').exists():
    try:
        if not archive.exists():
            urlretrieve('https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', archive)
        with ZipFile(archive) as z:
            z.extractall(DATA_DIR)
        print('Loaded OFFICIAL MovieLens dataset')
    except Exception as exc:
        print('Official download unavailable; using INCLUDED SYNTHETIC data:', str(exc)[:100])
if (official / 'ratings.csv').exists():
    ratings = pd.read_csv(official / 'ratings.csv')
    movies = pd.read_csv(official / 'movies.csv')
    DATA_SOURCE = 'MovieLens (real, anonymous ratings)'
else:
    ratings = pd.read_csv(DATA_DIR / 'synthetic_ratings.csv')
    movies = pd.read_csv(DATA_DIR / 'synthetic_movies.csv')
    DATA_SOURCE = 'Synthetic fallback (NOT real user ratings)'
print(DATA_SOURCE, 'ratings:', ratings.shape, 'movies:', movies.shape)
display(ratings.head(), movies.head())

### TODO 1 — Inspect the interaction data
1. Report the number of distinct users, movies with ratings, and observed ratings.
2. Compute sparsity = `1 − observed_ratings / (num_users × num_rated_movies)`.
3. Plot a histogram of ratings and a histogram of ratings per user.
4. In a markdown response, explain why an unrated item is **missing**, not a zero-star rating.

In [ ]:
# TODO 1: Complete all four tasks. Replace pass.
# n_users = ...
# n_movies = ...
# sparsity = ...
# plt.hist(...)
pass

## 1. Evaluation split and baselines
Hold out interactions **within each user**, preserving some training ratings for each user. A random holdout is instructional; real systems often require a time-based split. Evaluation metrics for observed ratings: RMSE and MAE.

In [ ]:
# Instructor-provided reproducible per-user split: 70% train, 15% validation, 15% test.
def split_user(group):
    ids = group.index.to_numpy().copy()
    np.random.default_rng(SEED + int(group.name)).shuffle(ids)
    n = len(ids); a = max(1, int(.70*n)); b = max(a+1, int(.85*n))
    return pd.Series(['train' if k<a else 'val' if k<b else 'test' for k in range(n)], index=ids)
parts = ratings.groupby('userId',group_keys=False).apply(split_user)
train = ratings.loc[parts[parts=='train'].index].copy()
val = ratings.loc[parts[parts=='val'].index].copy()
test = ratings.loc[parts[parts=='test'].index].copy()
assert len(train)+len(val)+len(test)==len(ratings)
print(len(train),len(val),len(test))

### TODO 2 — Baseline predictions
1. Compute the **global training mean** and the **per-user training mean**.
2. Predict validation ratings with each baseline. For users without a training mean, fall back to the global mean.
3. Clip predictions to the allowed rating range `[0.5, 5.0]`.
4. Report validation RMSE and MAE. Which baseline is better? Why?
5. **Check:** the number of predictions must equal `len(val)`.

In [ ]:
# TODO 2
# global_mean = ...
# user_means = ...
# global_pred = ...
# user_pred = ...
# print('RMSE:', ...)
pass

## 2. Collaborative filtering: matrix factorization
For user `u` and movie `i`, predict `ŷ_ui = μ + b_u + b_i + p_u · q_i`. Learn user and movie latent vectors with **observed ratings only**, using squared error and L2 regularization. This is a collaborative model: the latent features are learned from interactions, not movie genres.

In [ ]:
# Instructor scaffold: integer indexing, training arrays, and a lightweight SGD trainer.
user_ids = np.sort(ratings.userId.unique())
movie_ids = np.sort(ratings.movieId.unique())
u2i = {u:i for i,u in enumerate(user_ids)}
m2i = {m:i for i,m in enumerate(movie_ids)}
def arrays(df):
    return (df.userId.map(u2i).to_numpy(dtype=int),
            df.movieId.map(m2i).to_numpy(dtype=int),
            df.rating.to_numpy(dtype=float))
tu,ti,ty = arrays(train); vu,vi,vy = arrays(val); eu,ei,ey = arrays(test)
mu = float(ty.mean())
def predict_mf(U,V,bu,bi,users,items,mu=mu):
    return np.clip(mu+bu[users]+bi[items]+np.sum(U[users]*V[items],axis=1),.5,5.)
def fit_mf(k=12,lr=.01,reg=.05,epochs=12,seed=SEED):
    rg = np.random.default_rng(seed)
    U=rg.normal(0,.1,(len(user_ids),k)); V=rg.normal(0,.1,(len(movie_ids),k))
    bu=np.zeros(len(user_ids));bi=np.zeros(len(movie_ids)); history=[]
    for epoch in range(epochs):
        for t in rg.permutation(len(ty)):
            u,i,y=int(tu[t]),int(ti[t]),ty[t]
            err=mu+bu[u]+bi[i]+U[u]@V[i]-y
            pu=U[u].copy();qi=V[i].copy()
            bu[u]-=lr*(err+reg*bu[u]);bi[i]-=lr*(err+reg*bi[i])
            U[u]-=lr*(err*qi+reg*pu);V[i]-=lr*(err*pu+reg*qi)
        pred=predict_mf(U,V,bu,bi,vu,vi)
        history.append(np.sqrt(mean_squared_error(vy,pred)))
    return U,V,bu,bi,history
print('SGD scaffold ready')

### TODO 3 — Train collaborative filtering
1. Call `fit_mf(k=12, lr=0.01, reg=0.05, epochs=12)`.
2. Plot validation RMSE against epoch.
3. Report final validation RMSE and MAE.
4. Inspect the shapes of user matrix `U` and movie matrix `V` and explain what each row means.
5. **Check:** `U.shape[1] == V.shape[1] == 12`.

In [ ]:
# TODO 3
# U,V,bu,bi,history = fit_mf(...)
# ...
pass

### TODO 4 — Regularization and latent dimension
1. Run at least **four** experiments: `k ∈ {4, 12}` crossed with `reg ∈ {0.01, 0.2}` (8 epochs each is sufficient).
2. Make a DataFrame with `k`, `reg`, `val_RMSE`.
3. Select the best configuration **using validation only**; explain underfitting versus overfitting.
4. Train your selected configuration on the original training split; evaluate the held-out **test set once** after model selection.
5. **Check:** do not use `test` to choose hyperparameters.

In [ ]:
# TODO 4
# results = []
# for k in ...:
#     for reg in ...:
#         ...
pass

## 3. Mean normalization and cold start
A new user has no ratings. In **movie-mean normalization**, compute each movie's mean from the **training set only**; subtract it from observed training ratings and add it back at prediction time. For a completely new user, the movie mean is a reasonable fallback. This is different from **user-mean normalization**, which cannot compute a mean for an unrated new user.

### TODO 5 — New user (Eve) and mean normalization
1. Calculate each movie's mean training rating, with global-mean fallback for unrated movies.
2. Define a new user Eve with **no observed ratings**; do not invent ratings for her.
3. Recommend the top 10 movies by training movie mean, restricted to movies with at least 10 training ratings.
4. Explain why Eve's personalized latent vector cannot be learned from interactions alone and how content/profile information might help.
5. **Check:** no validation/test rating is used to compute a mean.

In [ ]:
# TODO 5
# movie_mean = train.groupby('movieId').rating.mean()
# movie_count = ...
# eligible = ...
# ...
pass

## 4. Content-based filtering with movie metadata
MovieLens provides `genres` and optionally `tags`; it does **not** contain user demographics. Build a **genre vector** for each movie, and a user's **preference profile** by averaging genre vectors of movies the user rated highly in the **training** set. This is a transparent content-based baseline.

In [ ]:
# Instructor scaffold: multi-hot genre vectors
movie_genres = movies.genres.fillna('(no genres listed)').str.split('|')
mlb = MultiLabelBinarizer()
X_movie = mlb.fit_transform(movie_genres).astype(float)
movie_row = {mid:i for i,mid in enumerate(movies.movieId)}
print('Movie feature matrix:',X_movie.shape,'genre labels:',mlb.classes_[:8])

### TODO 6 — Build and inspect content features
1. Display all genre labels and feature vector for the first movie.
2. Select one movie and retrieve five other movies with the **highest cosine similarity** of genre vectors; exclude itself.
3. Print titles, genres, and similarity scores.
4. Explain a limitation of genre-only similarity (e.g., movies sharing a broad genre but different themes).
5. **Check:** no self-recommendation.

In [ ]:
# TODO 6
# query_idx = 0
# scores = cosine_similarity(X_movie[query_idx:query_idx+1], X_movie).ravel()
# ...
pass

### TODO 7 — User preference profiles and recommendations
1. Pick an existing user with at least 15 training ratings.
2. Select their training movies rated **4 stars or higher**.
3. Average their genre vectors to create a preference vector; if there are no liked movies, use a safe fallback.
4. Compute cosine similarity between this profile and each movie vector.
5. Exclude **all training movies already rated by this user** and display the top 10 recommendations.
6. Explain why this model can score an entirely new movie with known genres.

In [ ]:
# TODO 7
# chosen_user = ...
# liked = train[(train.userId==chosen_user) & (train.rating>=4)]
# profile = ...
# ...
pass

## 5. Content-based two-tower neural networks (TensorFlow)
The lecture architecture learns user and item representations separately and combines them with a dot product. For this exercise, user inputs are **training-derived preference profiles**, optionally combined with **synthetic user attributes** when using the synthetic dataset. Movie inputs are multi-hot genres. Both towers output 32-dimensional vectors. The prediction can be `sigmoid(dot(v_user, v_movie))` for a binary liked/disliked target. **Important:** user profiles must be computed only from training interactions, and profile construction should exclude the target interaction when training a leakage-free production model. This simplified lab uses a fixed training-derived profile and is therefore an instructional demonstration, not a leakage-free benchmark for training-set accuracy.

In [ ]:
# Instructor scaffold: build per-user genre profiles from training data only.
profile = np.zeros((len(user_ids),X_movie.shape[1]))
for user_id,g in train.groupby('userId'):
    liked = g[g.rating>=4].movieId.tolist()
    idx=[movie_row[mid] for mid in liked if mid in movie_row]
    if idx: profile[u2i[user_id]] = X_movie[idx].mean(axis=0)
movie_features = np.stack([X_movie[movie_row[mid]] for mid in movie_ids])
print('Profiles:',profile.shape,'Movie features:',movie_features.shape)

### TODO 8 — Build the two-tower model
1. Import TensorFlow (install it first if necessary).
2. Create two `tf.keras.Input` objects: one for user profiles, one for movie genre vectors.
3. User tower: `Dense(64, relu)` → `Dense(32)`; movie tower: `Dense(128, relu)` → `Dense(32)`.
4. L2-normalize both output vectors using `tf.math.l2_normalize` in a Keras-compatible layer.
5. Compute their dot product; use a trainable scaling layer or sigmoid to produce a probability.
6. Compile with binary cross-entropy and Adam. **Check:** both embeddings have shape `(batch_size, 32)`.

In [ ]:
# TODO 8 
# import tensorflow as tf
# user_input = tf.keras.Input(shape=(profile.shape[1],))
# movie_input = tf.keras.Input(shape=(movie_features.shape[1],))
# ...
pass

### TODO 9 — Train, evaluate, and inspect embeddings
1. Convert observed ratings to binary labels using `rating >= 4` as positive; **ratings below 4 are observed negatives**, while unrated pairs are **unknown**, not negatives.
2. Build user/movie input pairs for `train` and `val` from `profile` and `movie_features`.
3. Train the two-tower model for up to 10 epochs with early stopping on validation loss.
4. Plot training/validation loss and report validation AUC (if both classes are present).
5. Extract 32-D movie embeddings and retrieve 5 nearest neighbors by cosine similarity.
6. Compare neighbors against the genre-only results in TODO 6. State a limitation of using fixed user profiles and why a leakage-free setup requires leave-one-out or out-of-fold profiles.

In [ ]:
# TODO 9 (optional)
# y_train_binary = (train.rating.to_numpy() >= 4).astype(int)
# ...
pass

## 6. Retrieval, ranking, and evaluation
A scalable recommender first **retrieves** a candidate set, then **ranks** it. Offline ranking metrics should be computed on held-out interactions. Because this dataset lacks exposure logs, unobserved movies are not confirmed negatives; the following ranking task is a simplified held-out-positive recovery evaluation.

### TODO 10 — Retrieval + ranking
1. For one user, retrieve the **50** highest-scoring unseen movies using the content-based score (TODO 7).
2. Re-rank those 50 candidates using the trained collaborative model (TODO 3), if available.
3. Show the top 10 titles and both scores.
4. Compare recommendation lists before and after re-ranking.
5. Discuss the trade-off between retrieving 20, 50, and 100 candidates and the latency/quality implications.
6. **Check:** no training-rated movie appears in the displayed list.

In [ ]:
# TODO 10
# candidate_ids = ...
# ...
pass

### TODO 11 — Evaluate held-out ranking quality
1. For at least 10 users, define **relevant test items** as held-out ratings `>=4`.
2. Score candidate movies that are **not in that user's training interactions**.
3. Compute `Recall@10 = (# relevant test movies in top 10) / (# relevant test movies)` for users with at least one relevant test item.
4. Average recall across users; compare content-based and collaborative approaches.
5. Explain why results can be optimistic/pessimistic without impression or exposure data and why RMSE and Recall@10 measure different goals.
6. **Check:** never use a user's test ratings to build their profile or select hyperparameters.

In [ ]:
# TODO 11
# recalls = []
# for user_id in ...:
#     relevant = set(test.loc[(test.userId==user_id) & (test.rating>=4), 'movieId'])
#     ...
pass

### TODO 12 — Ethical considerations and reflection
Answer in a Markdown cell:
1. How can optimizing click-through or watch time conflict with user welfare?
2. What is the **popularity bias** risk in collaborative filtering?
3. How would you address a cold-start user and a new movie fairly?
4. What privacy risks arise from using demographics, location, or viewing history? (The supplied synthetic profiles are illustrative, not real demographics.)
5. Suggest **two measurable safeguards** (e.g., diversity, harmful-content filtering, exposure fairness, user controls) and how you would evaluate them.

In [ ]:
# TODO 12: Write your reflection in a new Markdown cell below this one.
pass

## Submission checklist
- Complete TODOs 1–7 and 10–12; TODOs 8–9 are optional TensorFlow extensions unless assigned.
- Include at least three plots, one hyperparameter comparison table, and sample recommendations.
- Report **validation** and **test** metrics separately; explain data leakage prevention.
- State clearly whether your results use **real MovieLens** or **synthetic fallback**.
- Submit the completed `.ipynb` with outputs and a short (150–250 word) comparison of collaborative versus content-based filtering.

### References and dataset
- GroupLens MovieLens: https://grouplens.org/datasets/movielens/latest/
- Official dataset README: https://files.grouplens.org/datasets/movielens/ml-latest-small-README.html
- Official direct archive: https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
- TensorFlow MovieLens catalog: https://www.tensorflow.org/datasets/catalog/movielens

**Data note:** MovieLens Latest Small is intended for development/education rather than stable published benchmark comparisons. Read GroupLens license and attribution requirements before redistribution.